Script to count number of classes

Imports

In [21]:
import pandas as pd

df = pd.read_csv('C:/repos/Blood-Cell-Classification/LabelledData/LabelExport_20260317.csv')

classes_name = ['Monocyte without RBC', 'Monocyte with RBC', 'Unusable', 'RBC alone', 'Clustered cell', 'Lymphocyte?']

for name in classes_name:
    count = (df['choice'] == name).sum()
    print(f"{name} count: {count}")

print(f"Number of labelled cells: {len(df)-1}")


Monocyte without RBC count: 5053
Monocyte with RBC count: 961
Unusable count: 2478
RBC alone count: 567
Clustered cell count: 7203
Lymphocyte? count: 175
Number of labelled cells: 16436


In [7]:
# Check if there are any nan labels
df[df['choice'].isna()]['id'].tolist()

[]

In [13]:
df = pd.read_csv('C:/repos/Blood-Cell-Classification/LabelledData/LabelExport_20260317_CountingRBC.csv')

classes_name = [1, 2, 3]

for name in classes_name:
    count = (df['choice'] == name).sum()
    print(f"{name} count: {count}")

print(f"Number of labelled cells: {len(df)-1}")

1 count: 551
2 count: 242
3 count: 163
Number of labelled cells: 960


In [18]:
# Output the cumulative error
MonoNonMono = 0.9226
Clustered_class = 0.9381
Unclustered_RBCCount = 0.9375
Clustered_RBCCount = 0.9226
ThreeClass = 0.8634

RBCCount_Unclustered_Acc = Clustered_class * MonoNonMono * Unclustered_RBCCount
RBCCount_clustered_Acc = Clustered_class * MonoNonMono * Clustered_RBCCount 
ThreeClass_Clustered_Acc = ThreeClass * Clustered_RBCCount
ThreeClass_Unclustered_Acc = ThreeClass * Unclustered_RBCCount

print(f"Layered Classifiers Clustered: {RBCCount_clustered_Acc}")
print(f"Layered Classifiers Unclustered: {RBCCount_Unclustered_Acc}")
print(f"Three Class Clustered: {ThreeClass_Clustered_Acc}")
print(f"Three Class Unclustered: {ThreeClass_Unclustered_Acc}")

Layered Classifiers Clustered: 0.798502051956
Layered Classifiers Unclustered: 0.81139786875
Three Class Clustered: 0.79657284
Three Class Unclustered: 0.8094374999999999


In [27]:
import os
import json
from sklearn.model_selection import train_test_split

LABELLED_ROOT = r"D:/MMA_LabelledData/Sliced"
OUTPUT_SPLITS = r"D:/MMA_LabelledData/splits"

os.makedirs(OUTPUT_SPLITS, exist_ok=True)

# Collect all labeled filenames
all_files = []
for class_folder in os.listdir(LABELLED_ROOT):
    class_path = os.path.join(LABELLED_ROOT, class_folder)
    if not os.path.isdir(class_path):
        continue
    for fname in os.listdir(class_path):
        if fname.lower().endswith((".png", ".jpg", ".jpeg")):
            all_files.append(fname)

print("Total labeled images:", len(all_files))


Total labeled images: 16085


In [29]:
# Build labels for stratification
labels = []
for fname in all_files:
    for class_folder in os.listdir(LABELLED_ROOT):
        if os.path.isfile(os.path.join(LABELLED_ROOT, class_folder, fname)):
            labels.append(class_folder)
            break

# First split: train (70%) vs temp (30%)
train_files, temp_files, train_labels, temp_labels = train_test_split(
    all_files, labels, test_size=0.30, stratify=labels, random_state=42
)

# Second split: val (10%) vs test (20%)
val_files, test_files = train_test_split(
    temp_files, test_size=2/3, stratify=temp_labels, random_state=42
)

print(len(train_files), len(val_files), len(test_files))

splits = {
    "train": train_files,
    "val": val_files,
    "test": test_files
}

with open(os.path.join(OUTPUT_SPLITS, "dataset_splits.json"), "w") as f:
    json.dump(splits, f, indent=2)

print("Saved dataset_splits.json")



11259 1608 3218
Saved dataset_splits.json


In [39]:
import sys
import copy
import os

# keep your existing sys.path usage so imports work across folders
sys.path.append("C:/repos/Blood-Cell-Classification/Scripts/Validation")
from K_foldValidation import kfold_validate

sys.path.append("C:/repos/Blood-Cell-Classification/Scripts/Classifiers/Grid_Search")
from Classifier_MCvsNonMC import train, DEFAULT_CONFIG, load_samples, COLLAPSE_MAP

# optional: logging path or other project-specific paths
sys.path.append("C:/repos/Blood-Cell-Classification/Scripts/Logging")
# from Logger import setup_mlflow, end_run   # if you need to call logger functions directly

# prepare config
config = copy.deepcopy(DEFAULT_CONFIG)
config["architecture"]    = "resnet34"
config["data_dir"]        = r"D:\MMA_LabelledData\Sliced"   # set to your data root
config["checkpoint_path"] = r"checkpoints_stage1\resnet34_kfold.pth"
config["seed"]            = 42

# ensure checkpoint directory exists
os.makedirs(os.path.dirname(config["checkpoint_path"]), exist_ok=True)

# run k-fold validation
aggregate = kfold_validate(
    train_fn        = train,
    config          = config,
    load_fn         = load_samples,
    label_key       = "binary",        # or "raw" for stage2 / regression
    collapse_map    = COLLAPSE_MAP,    # only required when label_key="binary"
    experiment_name = "Stage1_KFold",
    notes           = "resnet34 baseline kfold",
    n_splits        = 5,
    results_csv     = None             # optional: path to save per-fold CSV
)

print(f"Final aggregate test_acc: {aggregate.get('test_acc_mean'):.4f} ± {aggregate.get('test_acc_std'):.4f}")



#################################################################
  Stratified 5-Fold Validation
  Architecture: resnet34
  Experiment:   Stage1_KFold
#################################################################

Loading all samples for k-fold split...
  ✓ MLflow tracking: sqlite:///D:/MMA_LabelledData/ML_logs/mlflow.db
  ✓ Experiment: Stage1_KFold

  Fold 1/5  —  Train+Val: 8056  |  Test: 2015

Training on: cuda  |  Architecture: resnet34
  ✓ MLflow tracking: sqlite:///D:/MMA_LabelledData/ML_logs/mlflow.db
  ✓ Experiment: Stage1_MCvsNonMC

  ✗ Fold 1 failed: Run with UUID 963bd875589344ecaca4b1dbeec25f6c is already active. To start a new run, first end the current run with mlflow.end_run(). To start a nested run, call start_run with nested=True

  Fold 2/5  —  Train+Val: 8057  |  Test: 2014

Training on: cuda  |  Architecture: resnet34
  ✓ MLflow tracking: sqlite:///D:/MMA_LabelledData/ML_logs/mlflow.db
  ✓ Experiment: Stage1_MCvsNonMC
  ✓ MLflow run started: 88d835da...

Datase

KeyboardInterrupt: 